## Section 5 — Classifier Training (Baseline vs. Augmented)

Train the same ImageNet-pretrained ResNet18 (frozen backbone, fresh 37-way FC head) three times, changing **only the training data** so the comparison in Section 6 isolates the effect of the synthetic images:

| Run | Training data |
|-----|---------------|
| `baseline` | real training images only (`train_idx` from 02) |
| `augmented` | real + **all** synthetic images (`manifests/augmented_train_index.csv` from 4.1) |
| `augmented_filtered` | real + **CLIP-filtered** synthetic images (`manifests/augmented_train_index_filtered.csv` from 4.2) |

Validation (real, from 02's split) and test (the untouched Oxford-IIIT Pet `test` split) are identical across runs, and every run is reseeded to `SEED` right before model init so the FC head and the shuffling order start out the same. This notebook assumes **02, 03 and 04 have already been run** and persisted their artifacts to disk — Section 5.1 only loads them.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from matplotlib import pyplot as plt
from PIL import Image
from torchvision.datasets import OxfordIIITPet
from tqdm import tqdm


def get_device() -> torch.device:
    """Resolve the best available torch device, preferring CUDA, then Apple MPS, then CPU.

    Returns:
        torch.device: the selected device for model inference/training.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_global_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch RNGs for reproducible runs.

    Args:
        seed: the seed value applied to all random number generators.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DATA_ROOT = Path("data")
MANIFESTS_DIR = Path("manifests")
MANIFESTS_DIR.mkdir(exist_ok=True)
SEED = 42
DEVICE = get_device()

set_global_seed(SEED)

print(f"Using device: {DEVICE}")

In [ ]:
import json
from dataclasses import dataclass

SPLITS_DIR = Path("splits")
TRAIN_VAL_SPLIT_PATH = SPLITS_DIR / "train_val_indices.json"  # written by 02_Captioning.ipynb (1.x)
AUGMENTED_INDEX_PATH = MANIFESTS_DIR / "augmented_train_index.csv"  # written by 04 (4.1)
FILTERED_AUGMENTED_INDEX_PATH = MANIFESTS_DIR / "augmented_train_index_filtered.csv"  # written by 04 (4.2)


@dataclass
class TrainingArtifacts:
    """Everything Section 5 needs from the on-disk outputs of notebooks 02 and 04.

    Attributes:
        trainval_dataset: Oxford-IIIT Pet `trainval` split, source of the real train/val images.
        test_dataset: Oxford-IIIT Pet `test` split, used untouched as the held-out test set.
        class_names: breed name for each class index, shared by both splits.
        trainval_labels: per-sample class id for every image in `trainval_dataset`.
        test_labels: per-sample class id for every image in `test_dataset`.
        train_idx: indices into `trainval_dataset` forming the training split (from 02).
        val_idx: indices into `trainval_dataset` forming the validation split (from 02).
        test_idx: indices into `test_dataset` — the whole split, kept as an array for symmetry.
        augmented_train_index: real training rows + every synthetic image (4.1).
        augmented_train_index_filtered: real training rows + CLIP-surviving synthetic images (4.2).
    """

    trainval_dataset: OxfordIIITPet
    test_dataset: OxfordIIITPet
    class_names: list[str]
    trainval_labels: np.ndarray
    test_labels: np.ndarray
    train_idx: np.ndarray
    val_idx: np.ndarray
    test_idx: np.ndarray
    augmented_train_index: pd.DataFrame
    augmented_train_index_filtered: pd.DataFrame


def get_labels(dataset: OxfordIIITPet) -> np.ndarray:
    """Extract the integer breed label for every sample in an OxfordIIITPet dataset.

    Reads the private `_labels` attribute when available (cheap) and falls back to iterating the
    dataset otherwise, which decodes every image and is therefore much slower.

    Args:
        dataset: the dataset to read labels from.

    Returns:
        np.ndarray: one integer class id per sample, in dataset order.
    """
    labels = getattr(dataset, "_labels", None)
    if labels is not None:
        return np.asarray(labels)
    return np.asarray([label for _, label in dataset])


def read_required_artifact(path: Path, produced_by: str) -> Path:
    """Assert that an upstream artifact exists, with an error naming the notebook that writes it.

    Args:
        path: the artifact file expected on disk.
        produced_by: human-readable name of the notebook/section that produces it.

    Returns:
        Path: `path` unchanged, so the call can be inlined into a read.

    Raises:
        FileNotFoundError: if `path` is missing, meaning the upstream notebook has not been run.
    """
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing — run {produced_by} first, it produces this artifact.")
    return path


def load_training_artifacts(
    data_root: Path,
    split_path: Path,
    augmented_index_path: Path,
    filtered_augmented_index_path: Path,
) -> TrainingArtifacts:
    """Load every upstream artifact Section 5 trains on into a single bundle.

    Nothing here is recomputed: the train/val split comes from 02 (so the synthetic images
    generated in 04 stay strictly inside the training half and never leak into validation) and the
    augmented indices come from 04. The test split is loaded straight from torchvision and is
    never touched by the pipeline.

    Args:
        data_root: root directory holding the downloaded Oxford-IIIT Pet dataset.
        split_path: JSON file with the `train_idx`/`val_idx` lists written by 02.
        augmented_index_path: CSV index of real + all synthetic training rows, written by 4.1.
        filtered_augmented_index_path: CSV index of real + filtered synthetic rows, written by 4.2.

    Returns:
        TrainingArtifacts: the loaded datasets, labels, split indices and augmented indices.

    Raises:
        FileNotFoundError: if any upstream artifact is missing.
    """
    trainval_dataset = OxfordIIITPet(
        root=str(data_root), split="trainval", target_types="category", download=False
    )
    test_dataset = OxfordIIITPet(root=str(data_root), split="test", target_types="category", download=False)

    split = json.loads(read_required_artifact(split_path, "02_Captioning.ipynb").read_text())

    return TrainingArtifacts(
        trainval_dataset=trainval_dataset,
        test_dataset=test_dataset,
        class_names=trainval_dataset.classes,
        trainval_labels=get_labels(trainval_dataset),
        test_labels=get_labels(test_dataset),
        train_idx=np.asarray(split["train_idx"]),
        val_idx=np.asarray(split["val_idx"]),
        test_idx=np.arange(len(test_dataset)),
        augmented_train_index=pd.read_csv(
            read_required_artifact(augmented_index_path, "04_ImageGeneration.ipynb (4.1)")
        ),
        augmented_train_index_filtered=pd.read_csv(
            read_required_artifact(filtered_augmented_index_path, "04_ImageGeneration.ipynb (4.2)")
        ),
    )


artifacts = load_training_artifacts(
    DATA_ROOT, TRAIN_VAL_SPLIT_PATH, AUGMENTED_INDEX_PATH, FILTERED_AUGMENTED_INDEX_PATH
)

trainval_dataset = artifacts.trainval_dataset
test_dataset = artifacts.test_dataset
class_names = artifacts.class_names
trainval_labels = artifacts.trainval_labels
test_labels = artifacts.test_labels
train_idx = artifacts.train_idx
val_idx = artifacts.val_idx
test_idx = artifacts.test_idx
augmented_train_index = artifacts.augmented_train_index
augmented_train_index_filtered = artifacts.augmented_train_index_filtered

print(
    f"Loaded {len(trainval_dataset)} trainval + {len(test_dataset)} test images over "
    f"{len(class_names)} classes\n"
    f"  splits          : train={len(train_idx)} | val={len(val_idx)} | test={len(test_idx)}\n"
    f"  augmented (4.1) : {len(augmented_train_index)} rows "
    f"({(augmented_train_index['source'] == 'synthetic').sum()} synthetic)\n"
    f"  filtered  (4.2) : {len(augmented_train_index_filtered)} rows "
    f"({(augmented_train_index_filtered['source'] == 'synthetic').sum()} synthetic)"
)

### 5.1 Building the train / validation / test datasets

Real and synthetic images are unified behind a single `ManifestImageDataset`: every row of an index frame is either a `real` sample (resolved by indexing back into the source `OxfordIIITPet` dataset) or a `synthetic` one (loaded from the JPEG written in 4.1). That way the baseline index, the two augmented indices from 04 and the val/test indices all feed the exact same `Dataset` class and the same deterministic transform — **no random augmentation anywhere**, so the only thing that differs between runs is which rows are in the training index.

In [ ]:
import time

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

CHECKPOINTS_DIR = Path("checkpoints")
REPORTS_DIR = Path("reports")
CHECKPOINTS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


In [ ]:
class ManifestImageDataset(Dataset):
    """Image classification dataset backed by a manifest of real and/or synthetic images.

    Each row of `index_df` is either a "real" sample (looked up by integer index into
    `source_dataset`) or a "synthetic" sample (loaded from an image file path). The same class
    serves the baseline training set, the augmented training set, and the val/test sets.
    """

    def __init__(self, index_df: pd.DataFrame, source_dataset: OxfordIIITPet, transform: transforms.Compose) -> None:
        """Build the dataset from an index manifest.

        Args:
            index_df: rows with `source` ("real"/"synthetic"), `class_id`, and `path` — for
                "real" rows `path` is the string form of an index into `source_dataset`; for
                "synthetic" rows `path` is an image file path.
            source_dataset: dataset that "real" rows' `path` indexes into.
            transform: preprocessing pipeline applied to every loaded PIL image.
        """
        self.index_df = index_df.reset_index(drop=True)
        self.source_dataset = source_dataset
        self.transform = transform

    def __len__(self) -> int:
        return len(self.index_df)

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, int]:
        row = self.index_df.iloc[idx]
        if row["source"] == "real":
            image, _ = self.source_dataset[int(row["path"])]
        else:
            image = Image.open(row["path"]).convert("RGB")
        return self.transform(image), int(row["class_id"])

In [ ]:
def build_eval_transform(image_size: int) -> transforms.Compose:
    """Build the image preprocessing pipeline shared by every split and run.

    The same deterministic transform (no random augmentation) is used for train, val and test
    across both Run A and Run B, so the only difference between runs is the training data itself,
    not an extra source of stochastic augmentation.

    Args:
        image_size: target square size (in pixels) images are resized to.

    Returns:
        transforms.Compose: resize -> tensor -> ImageNet normalization pipeline.
    """
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ]
    )


IMAGE_TRANSFORM = build_eval_transform(IMAGE_SIZE)


In [ ]:
def build_real_index(indices: np.ndarray, labels: np.ndarray, class_names: list[str]) -> pd.DataFrame:
    """Build a `{source, class_id, class_label, path}` index for a set of real-dataset indices.

    `path` stores the string form of each index rather than a file path: real images are resolved
    by indexing back into their source `OxfordIIITPet` dataset (see `ManifestImageDataset` above),
    not loaded from disk directly. This matches the schema `04_ImageGeneration.ipynb` used for the
    real half of the augmented indices, so a baseline index built here and an augmented index read
    from `manifests/` are interchangeable as far as `ManifestImageDataset` is concerned.

    Args:
        indices: dataset indices (into whichever dataset `labels` was computed from).
        labels: full per-sample label array for that dataset (`get_labels` output), indexed by
            `indices`.
        class_names: breed name for each class index.

    Returns:
        pd.DataFrame: one "real" row per index, with `source="real"`.
    """
    class_ids = labels[indices]
    return pd.DataFrame(
        {
            "source": "real",
            "class_id": class_ids,
            "class_label": [class_names[label] for label in class_ids],
            "path": indices.astype(str),
        }
    )

In [ ]:
baseline_train_index = build_real_index(train_idx, trainval_labels, class_names)
val_index = build_real_index(val_idx, trainval_labels, class_names)
test_index = build_real_index(test_idx, test_labels, class_names)

print(
    f"baseline_train_index: {len(baseline_train_index)} | val_index: {len(val_index)} | "
    f"test_index: {len(test_index)}"
)


In [ ]:
baseline_train_dataset = ManifestImageDataset(baseline_train_index, trainval_dataset, IMAGE_TRANSFORM)
augmented_train_dataset = ManifestImageDataset(augmented_train_index, trainval_dataset, IMAGE_TRANSFORM)
augmented_filtered_train_dataset = ManifestImageDataset(augmented_train_index_filtered, trainval_dataset, IMAGE_TRANSFORM)
val_eval_dataset = ManifestImageDataset(val_index, trainval_dataset, IMAGE_TRANSFORM)
test_eval_dataset = ManifestImageDataset(test_index, test_dataset, IMAGE_TRANSFORM)

print(
    f"Run A (baseline) train: {len(baseline_train_dataset)} | "
    f"Run B (augmented) train: {len(augmented_train_dataset)} | "
    f"val: {len(val_eval_dataset)} | test: {len(test_eval_dataset)}"
)


### 5.2 Model and training loop

ResNet18 pretrained on ImageNet with **every backbone parameter frozen** and only a fresh 37-way `fc` head trainable — with ~2.9k real training images the frozen-backbone setup is both the cheap option locally and the one that makes the augmentation effect easiest to read, since the feature extractor is identical across runs.

Training uses Adam on the trainable parameters only, plus early stopping on validation loss: `EarlyStopping` checkpoints the model on every improvement and flags a stop after `patience` flat epochs, and `train_classifier` reloads that checkpoint at the end so the returned model always carries the best-validation weights rather than the last epoch's. Progress is reported at two levels — a live `tqdm` bar with running loss/accuracy inside each epoch, and a one-line summary per epoch with the timing and the early-stopping state.

In [ ]:
def build_classifier(num_classes: int, device: torch.device) -> nn.Module:
    """Build an ImageNet-pretrained ResNet18 with a frozen backbone and a fresh FC head.

    Args:
        num_classes: number of output classes for the replaced final layer.
        device: device to move the model to.

    Returns:
        nn.Module: ResNet18 with every parameter frozen except the new `fc` layer.
    """
    model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    for param in model.parameters():
        param.requires_grad = False
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    optimizer: torch.optim.Optimizer | None = None,
    description: str = "",
) -> tuple[float, float]:
    """Run one training (if `optimizer` is given) or evaluation epoch over `loader`.

    A `tqdm` bar tracks batch progress and shows the running loss/accuracy so a long epoch can be
    monitored live rather than only reporting once it has finished; the bar is transient
    (`leave=False`) so the persistent log stays the one-line-per-epoch summary printed by
    `train_classifier`.

    Args:
        model: classifier to run.
        loader: DataLoader yielding (images, labels) batches.
        criterion: loss function.
        device: device the model and batches live on.
        optimizer: if provided, the epoch trains (backward + step); otherwise it only evaluates.
        description: label shown on the progress bar, e.g. "baseline e3/8 train".

    Returns:
        tuple[float, float]: (average loss, accuracy) over the epoch.
    """
    is_train = optimizer is not None
    model.train(is_train)

    total_loss, correct, total = 0.0, 0, 0
    progress = tqdm(loader, desc=description, unit="batch", leave=False)
    with torch.set_grad_enabled(is_train):
        for images, labels in progress:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += images.size(0)
            progress.set_postfix(loss=f"{total_loss / total:.4f}", acc=f"{correct / total:.4f}")

    return total_loss / total, correct / total

In [ ]:
class EarlyStopping:
    """Monitors validation loss during training and signals when to stop.

    Checkpoints `model`'s state dict to `checkpoint_path` whenever validation loss improves by
    more than `min_delta`. After `patience` consecutive epochs without improvement, `early_stop`
    is set and the caller should stop training; reloading `checkpoint_path` afterwards restores
    the best-validation-loss weights rather than the final epoch's. `improved`, `counter` and
    `best_epoch` are exposed so the training loop can report *why* it kept going or stopped.
    """

    def __init__(self, checkpoint_path: Path, patience: int = 5, min_delta: float = 0.0) -> None:
        """
        Args:
            checkpoint_path: destination `.pt` file for the state dict on each improvement.
            patience: consecutive non-improving epochs to tolerate before signalling stop.
            min_delta: minimum decrease in validation loss to count as an improvement.
        """
        self.checkpoint_path = checkpoint_path
        self.patience = patience
        self.min_delta = min_delta
        self.best_val_loss: float | None = None
        self.best_epoch: int | None = None
        self.counter = 0
        self.improved = False
        self.early_stop = False

    def __call__(self, val_loss: float, model: nn.Module, epoch: int) -> None:
        """Update state with this epoch's validation loss, checkpointing `model` on improvement.

        Args:
            val_loss: validation loss for the epoch just completed.
            model: model to checkpoint if `val_loss` is a new best.
            epoch: 1-based number of the epoch just completed, recorded as `best_epoch` on
                improvement so the caller can report where the restored weights came from.
        """
        if self.best_val_loss is None or (self.best_val_loss - val_loss) > self.min_delta:
            self.best_val_loss = val_loss
            self.best_epoch = epoch
            self.counter = 0
            self.improved = True
            torch.save(model.state_dict(), self.checkpoint_path)
        else:
            self.counter += 1
            self.improved = False
            if self.counter >= self.patience:
                self.early_stop = True

    def describe_last_epoch(self) -> str:
        """Summarise the most recent `__call__` for the per-epoch training log.

        Returns:
            str: either the improvement + checkpoint confirmation, or the patience counter.
        """
        if self.improved:
            return f"val_loss improved -> saved {self.checkpoint_path.name}"
        return f"no improvement (patience {self.counter}/{self.patience})"


def select_restored_epoch(history: pd.DataFrame) -> pd.Series:
    """Return the history row holding the weights the run actually restored.

    Read-side counterpart of `EarlyStopping`: `train_classifier` marks that epoch with a
    `restored` flag, which is *not* always the arg-min of `val_loss` — with `min_delta > 0` a
    later epoch can post a nominally lower loss without clearing the improvement threshold, so it
    is never checkpointed. Falls back to the arg-min for histories written before the flag
    existed, where the two coincide because `min_delta` was 0.

    Args:
        history: per-epoch history frame for a single run.

    Returns:
        pd.Series: the row whose weights were checkpointed and reloaded at the end of the run.
    """
    if "restored" in history.columns and history["restored"].any():
        return history.loc[history["restored"].idxmax()]
    return history.loc[history["val_loss"].idxmin()]

In [ ]:
def train_classifier(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    epochs: int,
    lr: float,
    run_name: str,
    checkpoint_path: Path,
    patience: int = 5,
    min_delta: float = 0.0,
) -> pd.DataFrame:
    """Train `model` for up to `epochs` epochs, stopping early on stalled validation loss.

    Only parameters with `requires_grad=True` (the replaced FC layer) receive gradient updates,
    per the plan's frozen-backbone requirement. `EarlyStopping` checkpoints `model` to
    `checkpoint_path` on every validation-loss improvement and flags a stop after `patience`
    non-improving epochs; once training ends (by exhausting `epochs` or stopping early), the
    best-validation-loss weights are reloaded into `model` so it never reflects an overfit final
    epoch.

    Logging is deliberately verbose so a run can be followed live: a header with the run's data
    sizes and hyperparameters, a transient `tqdm` bar per train/val pass (see `run_epoch`), one
    summary line per epoch with losses, accuracies, wall time and early-stopping state, and a
    closing line naming the epoch whose weights were restored.

    Args:
        model: classifier to train, already moved to `device`.
        train_loader: training DataLoader.
        val_loader: validation DataLoader.
        device: device the model and batches live on.
        epochs: maximum number of training epochs.
        lr: learning rate for the Adam optimizer.
        run_name: label used in the printed progress lines and the `run` column.
        checkpoint_path: destination `.pt` file for the best-validation-loss state dict.
        patience: consecutive non-improving epochs to tolerate before stopping early.
        min_delta: minimum decrease in validation loss to count as an improvement.

    Returns:
        pd.DataFrame: one row per completed epoch with `run, epoch, train_loss, train_acc,
        val_loss, val_acc, epoch_seconds, restored` — `restored` marking the single epoch whose
        checkpoint was reloaded into `model` (see `select_restored_epoch`).
    """
    criterion = nn.CrossEntropyLoss()
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.Adam(trainable, lr=lr)
    early_stopping = EarlyStopping(checkpoint_path, patience=patience, min_delta=min_delta)

    n_trainable = sum(p.numel() for p in trainable)
    print(f"\n{'=' * 92}")
    print(f"RUN '{run_name}' | device={device} | {n_trainable:,} trainable params")
    print(
        f"  data   : train={len(train_loader.dataset):>5} imgs / {len(train_loader):>3} batches | "
        f"val={len(val_loader.dataset):>5} imgs / {len(val_loader):>3} batches"
    )
    print(
        f"  hparams: max_epochs={epochs} | lr={lr} | batch_size={train_loader.batch_size} | "
        f"early_stopping(patience={patience}, min_delta={min_delta})"
    )
    print(f"{'=' * 92}", flush=True)

    run_start = time.time()
    history = []
    for epoch in range(1, epochs + 1):
        epoch_start = time.time()
        train_loss, train_acc = run_epoch(
            model, train_loader, criterion, device, optimizer, f"{run_name} e{epoch}/{epochs} train"
        )
        val_loss, val_acc = run_epoch(
            model, val_loader, criterion, device, description=f"{run_name} e{epoch}/{epochs} val"
        )
        epoch_seconds = time.time() - epoch_start

        early_stopping(val_loss, model, epoch)
        history.append(
            {"run": run_name, "epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
             "val_loss": val_loss, "val_acc": val_acc, "epoch_seconds": epoch_seconds}
        )
        print(
            f"[{run_name}] epoch {epoch:>2}/{epochs} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} | "
            f"{epoch_seconds:5.1f}s | {early_stopping.describe_last_epoch()}",
            flush=True,
        )

        if early_stopping.early_stop:
            print(f"[{run_name}] early stop: {patience} epochs without a val_loss improvement")
            break

    print(
        f"[{run_name}] finished {len(history)} epochs in {time.time() - run_start:.1f}s | "
        f"best val_loss {early_stopping.best_val_loss:.4f} @ epoch {early_stopping.best_epoch} | "
        f"restoring those weights from {checkpoint_path}",
        flush=True,
    )
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    history_df = pd.DataFrame(history)
    history_df["restored"] = history_df["epoch"] == early_stopping.best_epoch
    return history_df

In [ ]:
def get_or_train_classifier(
    checkpoint_path: Path,
    history_path: Path,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: torch.device,
    num_classes: int,
    epochs: int,
    lr: float,
    seed: int,
    run_name: str,
    patience: int = 5,
    min_delta: float = 0.0,
) -> tuple[nn.Module, pd.DataFrame]:
    """Reuse a trained classifier + history from disk if present, otherwise train and save both.

    Reseeds with `seed` right before building the model so every run starts from identical initial
    FC weights and DataLoader shuffling — the only difference between runs is the training data
    itself. `checkpoint_path` doubles as the early-stopping checkpoint target (see
    `train_classifier`), so a freshly trained model always holds the best-validation-loss weights,
    not necessarily those from the final epoch.

    Args:
        checkpoint_path: destination/source `.pt` file for the trained model's state dict.
        history_path: destination/source CSV file for the per-epoch training history.
        train_loader: training DataLoader for this run.
        val_loader: validation DataLoader (shared across runs).
        device: device to train on.
        num_classes: number of output classes.
        epochs: maximum number of training epochs, identical across runs for a fair comparison.
        lr: learning rate, identical across runs for a fair comparison.
        seed: seed reapplied before model init, for reproducibility and cross-run parity.
        run_name: label for progress printing and the `run` column in the history.
        patience: consecutive non-improving epochs to tolerate before stopping early.
        min_delta: minimum decrease in validation loss to count as an improvement.

    Returns:
        tuple[nn.Module, pd.DataFrame]: the trained model (on `device`) and its training history.
    """
    set_global_seed(seed)
    model = build_classifier(num_classes, device)

    if checkpoint_path.exists() and history_path.exists():
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        history = pd.read_csv(history_path)
        best = select_restored_epoch(history)
        print(
            f"Loaded cached '{run_name}' run from {checkpoint_path} — {len(history)} epochs, "
            f"restored epoch {int(best['epoch'])} (val_loss {best['val_loss']:.4f}, "
            f"val_acc {best['val_acc']:.4f}); skipped training."
        )
        return model, history

    history = train_classifier(
        model, train_loader, val_loader, device, epochs, lr, run_name, checkpoint_path, patience, min_delta
    )
    history.to_csv(history_path, index=False)
    print(f"Saved '{run_name}' history to {history_path}")
    return model, history

### 5.3 The three runs

Same hyperparameters, same seed, same validation loader — only `train_loader` changes. Each run is cached by its `(checkpoint, history)` pair, so re-running the notebook after a kernel restart reloads the trained head instead of retraining; delete the files under `checkpoints/` and `reports/` to force a fresh run.

In [ ]:
BATCH_SIZE = 32
TRAIN_EPOCHS = 15  # upper bound only — early stopping decides where each run actually ends
LEARNING_RATE = 1e-3
NUM_CLASSES = len(class_names)
# Patience must stay well below TRAIN_EPOCHS, otherwise the ceiling is always hit first and early
# stopping never gets to fire; 3 flat epochs out of a 15-epoch budget is a real stopping criterion.
EARLY_STOPPING_PATIENCE = 3  # consecutive non-improving epochs tolerated before stopping
EARLY_STOPPING_MIN_DELTA = 0.0  # minimum val_loss decrease to count as an improvement

val_loader = DataLoader(val_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

baseline_train_loader = DataLoader(baseline_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
augmented_train_loader = DataLoader(augmented_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
augmented_filtered_train_loader = DataLoader(
    augmented_filtered_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

In [ ]:
baseline_model, baseline_history = get_or_train_classifier(
    CHECKPOINTS_DIR / "baseline_resnet18.pt",
    REPORTS_DIR / "baseline_history.csv",
    baseline_train_loader,
    val_loader,
    DEVICE,
    NUM_CLASSES,
    TRAIN_EPOCHS,
    LEARNING_RATE,
    SEED,
    "baseline",
    EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_MIN_DELTA,
)


In [ ]:
augmented_model, augmented_history = get_or_train_classifier(
    CHECKPOINTS_DIR / "augmented_resnet18.pt",
    REPORTS_DIR / "augmented_history.csv",
    augmented_train_loader,
    val_loader,
    DEVICE,
    NUM_CLASSES,
    TRAIN_EPOCHS,
    LEARNING_RATE,
    SEED,
    "augmented",
    EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_MIN_DELTA,
)


In [ ]:
augmented_filtered_model, augmented_filtered_history = get_or_train_classifier(
    CHECKPOINTS_DIR / "augmented_filtered_resnet18.pt",
    REPORTS_DIR / "augmented_filtered_history.csv",
    augmented_filtered_train_loader,
    val_loader,
    DEVICE,
    NUM_CLASSES,
    TRAIN_EPOCHS,
    LEARNING_RATE,
    SEED,
    "augmented_filtered",
    EARLY_STOPPING_PATIENCE,
    EARLY_STOPPING_MIN_DELTA,
)

### 5.4 Learning curves

All three runs on one pair of axes — loss on the left, accuracy on the right, one colour per run with **dashed = train** and **solid = validation**, and a ★ marking the epoch whose weights early stopping restored. Overlaying the runs (rather than one figure each) is what makes the augmentation effect readable: a useful synthetic set should pull the *validation* curves apart while the train curves stay comparable, whereas a train/val gap that widens for the augmented runs only means the extra images are being memorised rather than generalising.

The per-run histories are also concatenated into `reports/training_histories.csv` for Section 6.

In [ ]:
def plot_learning_curves(histories: dict[str, pd.DataFrame], output_path: Path) -> plt.Figure:
    """Overlay the train/val loss and accuracy curves of every run on a shared pair of axes.

    One colour per run, dashed for train and solid for validation, with a star on the epoch whose
    weights early stopping restored. The legend is drawn once at figure level (the styling is
    identical in both panels) so it never occludes the curves.

    Args:
        histories: mapping of run name -> per-epoch history frame with `epoch`, `train_loss`,
            `val_loss`, `train_acc` and `val_acc` columns, as returned by
            `get_or_train_classifier`.
        output_path: PNG file the figure is written to.

    Returns:
        plt.Figure: the rendered figure, so the caller can further annotate or re-save it.
    """
    fig, (loss_ax, acc_ax) = plt.subplots(1, 2, figsize=(15, 5.5))
    palette = plt.get_cmap("tab10")
    max_epoch = max(int(history["epoch"].max()) for history in histories.values())

    for position, (run_name, history) in enumerate(histories.items()):
        color = palette(position)
        restored = select_restored_epoch(history)
        for ax, train_col, val_col in (
            (loss_ax, "train_loss", "val_loss"),
            (acc_ax, "train_acc", "val_acc"),
        ):
            ax.plot(history["epoch"], history[train_col], color=color, linestyle="--",
                    marker="o", markersize=3, alpha=0.6, label=f"{run_name} — train")
            ax.plot(history["epoch"], history[val_col], color=color, linestyle="-",
                    marker="o", markersize=4, label=f"{run_name} — val")
            ax.scatter([restored["epoch"]], [restored[val_col]], color=color, marker="*",
                       s=240, zorder=5, edgecolors="white", linewidths=0.7)

    for ax, ylabel, title in (
        (loss_ax, "cross-entropy loss", "Loss per epoch"),
        (acc_ax, "accuracy", "Accuracy per epoch"),
    ):
        ax.set_xlabel("epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.set_xticks(range(1, max_epoch + 1))
        ax.grid(alpha=0.3)

    fig.suptitle(
        "Learning curves — dashed = train, solid = validation, ★ = restored (best val_loss) epoch",
        fontsize=13,
    )
    # Reserve a band at the bottom for the shared legend and at the top for the suptitle, so
    # neither ends up drawn over the curves.
    fig.tight_layout(rect=(0.0, 0.09, 1.0, 0.94))
    handles, labels = loss_ax.get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(histories), fontsize=9, frameon=False)

    fig.savefig(output_path, dpi=150)
    print(f"Saved learning-curve figure to {output_path}")
    return fig


def summarise_runs(histories: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Condense each run's history into the single epoch whose weights it restored.

    Args:
        histories: mapping of run name -> per-epoch history frame (see `plot_learning_curves`).

    Returns:
        pd.DataFrame: one row per run with the epochs trained, the restored epoch and its
        train/val losses and accuracies, plus total wall-clock training time.
    """
    rows = []
    for run_name, history in histories.items():
        restored = select_restored_epoch(history)
        rows.append(
            {
                "run": run_name,
                "epochs_trained": len(history),
                "restored_epoch": int(restored["epoch"]),
                "train_loss": restored["train_loss"],
                "train_acc": restored["train_acc"],
                "val_loss": restored["val_loss"],
                "val_acc": restored["val_acc"],
                "total_seconds": history["epoch_seconds"].sum() if "epoch_seconds" in history else float("nan"),
            }
        )
    return pd.DataFrame(rows).set_index("run")


run_histories = {
    "baseline": baseline_history,
    "augmented": augmented_history,
    "augmented_filtered": augmented_filtered_history,
}

pd.concat(run_histories.values(), ignore_index=True).to_csv(REPORTS_DIR / "training_histories.csv", index=False)
plot_learning_curves(run_histories, REPORTS_DIR / "learning_curves.png")
plt.show()

run_summary = summarise_runs(run_histories)
print("\nRestored-epoch summary (the weights each run carries into Section 6):")
run_summary

In [ ]:
print("Section 5 complete — artifacts on disk:")
for run_name, history in run_histories.items():
    restored = select_restored_epoch(history)
    print(
        f"  {CHECKPOINTS_DIR}/{run_name}_resnet18.pt : {len(history)} epochs, "
        f"restored epoch {int(restored['epoch'])} (val_acc {restored['val_acc']:.4f})"
    )
print(f"  {REPORTS_DIR}/training_histories.csv : per-epoch history for all {len(run_histories)} runs")
print(f"  {REPORTS_DIR}/learning_curves.png : the figure above")
print("\nNext: run 06_Evaluation&Comparison.ipynb.")